# Exercise for lecture 6: Build and test a CDSS yourself

**Health Information Systems · winter term 2026/27**

In the lecture we met the CDSS chain:

**DATA → TRIGGER → LOGIC → OUTPUT → ACTION → FEEDBACK**

Today you build the middle part yourself: a **rule**, a **model**, a **threshold** – and you check whether the whole thing
still works in a second hospital.

| Part | Topic | approx. time |
|---|---|---:|
| 0 | Setup | 5 min |
| 1 | Getting to know the data | 10 min |
| 2 | A rule as a function – with tests | 15 min |
| 3 | Training a model | 10 min |
| 4 | Choosing the threshold | 15 min |
| 5 | A suspiciously good variable | 8 min |
| 6 | External validation: hospital B | 15 min |
| 7 | What does a missing value mean? | 7 min |
| 8 | From model to CDSS | 5 min |

> ⚠️ **Important:** the data set is **fully synthetic**. The outcome `verschlechterung_6h` is a “clinical deterioration
> within 6 hours” defined for teaching – **not a sepsis diagnosis**.
> What you build today is a **teaching prototype**, not a clinically usable system.

## Part 0 – Setup (5 min)

Run the following cell. It loads the libraries and defines a helper function `metrics()`
that we will need again and again.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.calibration import calibration_curve

pd.set_option("display.precision", 2)


def metrics(y_true, y_alert):
    """Computes the key metrics of an alert (0/1) against the actual outcome (0/1)."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_alert, labels=[0, 1]).ravel()
    return {
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "Sensitivity": tp / (tp + fn) if tp + fn else np.nan,
        "Specificity": tn / (tn + fp) if tn + fp else np.nan,
        "PPV": tp / (tp + fp) if tp + fp else np.nan,
        "Alerts per 100 pat.": 100 * (tp + fp) / len(y_true),
    }

print("Setup ok")

## Part 1 – Getting to know the data (10 min)

Two hospitals, the same variables. Each row is one patient at the **time of prediction**.

The column names are German (the data set is shared with the German version of this exercise):

| Column | Meaning |
|---|---|
| `patient_id`, `klinik` | patient ID, hospital (A or B) |
| `alter` | age (years) |
| `herzfrequenz` | heart rate (/min) |
| `atemfrequenz` | respiratory rate (/min) |
| `rr_systolisch` | systolic blood pressure (mmHg) |
| `temperatur` | temperature (°C) |
| `spo2` | oxygen saturation (%) |
| `kreatinin` | creatinine (mg/dl) |
| `leukozyten` | white blood cell count (10³/µl) |
| `crp` | C-reactive protein (mg/l) |
| `laktat` | lactate (mmol/l) |
| `bewusstsein_veraendert` | altered mentation (0/1) |
| `antibiotikum_gegeben` | antibiotic given (0/1) |
| `verschlechterung_6h` | **outcome:** deterioration within 6 hours (0/1) |

In [ ]:
a = pd.read_csv("klinik_A.csv")
b = pd.read_csv("klinik_B.csv")
print("Hospital A:", a.shape, " Hospital B:", b.shape)
a.head()

In [ ]:
# Task 1a: What is the event rate (share of verschlechterung_6h == 1) in hospital A and in hospital B?
# Hint: .mean() on a 0/1 column gives the share.
rate_a = None  # TODO
rate_b = None  # TODO
assert rate_a is not None and rate_b is not None, "Please calculate rate_a and rate_b first."
print(f"Event rate A: {rate_a:.1%}   B: {rate_b:.1%}")

In [ ]:
# A look at the distributions
a.describe().T[["mean", "std", "min", "max"]]

**Task 1b (discussion):** Go through the columns. Which information would **actually be available** at the time
of prediction – and which are you unsure about? Note your guess; we come back to it in part 5.

*Your answer:* …

## Part 2 – A rule as a function, with tests (15 min)

We implement a **qSOFA-like rule** (simplified for teaching):

- respiratory rate ≥ 22/min → 1 point
- systolic blood pressure ≤ 100 mmHg → 1 point
- altered mentation → 1 point
- **Alert if at least 2 points**

**Task 2a:** Complete the function.

In [ ]:
def qsofa_alert(atemfrequenz, rr_systolisch, bewusstsein_veraendert):
    """Returns True if at least 2 of the 3 criteria are met."""
    punkte = 0
    # TODO: award points
    ...
    return punkte >= 2

**Task 2b – Tests:** A rule is software – and software gets tested. The first two tests are given
(Maria K. from the lecture and an unremarkable patient). Add **two tests of your own for the boundary values**
(what happens exactly at a respiratory rate of 22 and a blood pressure of 100?).

In [ ]:
# Reference cases
assert qsofa_alert(atemfrequenz=26, rr_systolisch=96, bewusstsein_veraendert=0) == True    # Maria K.
assert qsofa_alert(atemfrequenz=18, rr_systolisch=125, bewusstsein_veraendert=0) == False  # unremarkable

# TODO: two boundary-value tests
# assert qsofa_alert(...) == ...
# assert qsofa_alert(...) == ...
print("All tests passed")

**Task 2c – Regression test:** A colleague suggests changing the respiratory rate threshold to `> 22`.
Change the function to try it and run the test cell again. Which test fails – and what do you learn from it?
(Afterwards, please change it back to `>= 22`.)

*Your answer:* …

**Task 2d:** Apply the rule to all patients in hospital A and compute the metrics.

In [ ]:
a["alert_regel"] = a.apply(
    lambda r: qsofa_alert(r["atemfrequenz"], r["rr_systolisch"], r["bewusstsein_veraendert"]), axis=1)

# TODO: metrics of the rule in hospital A
# metrics(...)

**Question:** The specificity is high, the sensitivity low. Does that fit the statement from the lecture that qSOFA
is **not recommended as a single screening tool**?

*Your answer:* …

## Part 3 – Training a model (10 min)

Now we learn the decision function from data: a **logistic regression** on hospital A
(70 % training, 30 % test).

In [ ]:
FEATURES = ["alter", "herzfrequenz", "atemfrequenz", "rr_systolisch", "temperatur", "spo2",
            "kreatinin", "leukozyten", "crp", "bewusstsein_veraendert"]
TARGET = "verschlechterung_6h"

train_a, test_a = train_test_split(a, test_size=0.3, random_state=42, stratify=a[TARGET])

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(train_a[FEATURES], train_a[TARGET])

# The model outputs probabilities, not a yes/no decision:
p_test_a = model.predict_proba(test_a[FEATURES])[:, 1]
p_test_a[:5]

In [ ]:
# Task 3a: Compute the model's AUC on the test data of hospital A.
auc_a = None  # TODO
assert auc_a is not None, "Please calculate auc_a first."
print(f"AUC hospital A (test): {auc_a:.3f}")

**Question:** The AUC tells you how well the model *ranks* patients with and without the event. Can you
trigger an alert with an AUC alone? What is still missing?

*Your answer:* …

## Part 4 – Choosing the threshold (15 min)

**Task 4a:** Compute the metrics for the thresholds 0.1 to 0.8.

In [ ]:
rows = []
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    # TODO: alert = probability >= t; compute the metrics and append them with the threshold to 'rows'
    # Format of each row: a dictionary with the key "Threshold" plus the metrics,
    # e.g. rows.append({"Threshold": t, **metrics(..., ...)})
    ...
assert rows, "Please complete the loop first (see the format above)."
table = pd.DataFrame(rows).set_index("Threshold")
table

In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(table.index, table["Sensitivity"], marker="o", label="Sensitivity")
ax1.plot(table.index, table["PPV"], marker="o", label="PPV")
ax1.set_xlabel("Threshold"); ax1.set_ylabel("Share"); ax1.set_ylim(0, 1)
ax2 = ax1.twinx()
ax2.bar(table.index, table["Alerts per 100 pat."], width=0.04, alpha=0.25, color="grey", label="Alerts/100")
ax2.set_ylabel("Alerts per 100 patients")
ax1.legend(loc="upper center"); plt.title("Hospital A: what the threshold changes"); plt.show()

**Task 4b – Your decision:** The ward says: *“We no longer take more than 15 alerts per 100 patients
seriously.”* Which threshold do you choose? What are the sensitivity and PPV then?
And: who should actually make this decision?

*Your answer:* …

In [ ]:
# Set the threshold you chose (it is reused in part 6)
THRESHOLD = None  # enter your decision from task 4b here
assert THRESHOLD is not None, "Please enter your threshold from task 4b first."
print("THRESHOLD =", THRESHOLD)

## Part 5 – A suspiciously good variable (8 min)

**Task 5a:** Train the same model again, this time **additionally** with `antibiotikum_gegeben` (antibiotic given).
How does the AUC change?

In [ ]:
FEATURES_PLUS = FEATURES + ["antibiotikum_gegeben"]
# TODO: train the model 'model_leak' and compute the AUC on test_a


**Task 5b:** The model becomes much “better”. Would you roll it out like this? Answer the question:
**When was this information available?**

*Your answer:* …

## Part 6 – External validation: hospital B (15 min)

Your model (without the leakage variable) is used in hospital B. There, patients are older and
deteriorations are rarer.

**Task 6a:** Compute the AUC and the metrics at your `THRESHOLD` for hospital B and compare them with hospital A.

In [ ]:
p_b = model.predict_proba(b[FEATURES])[:, 1]
# TODO: AUC for hospital B; metrics at THRESHOLD for A (test) and B side by side


**Task 6b – Calibration:** The AUC is almost the same. But are the predicted risks right?
Compare the mean prediction with the observed rate, and draw the calibration curves.

In [ ]:
print(f"Hospital B – mean predicted risk: {p_b.mean():.1%}   observed rate: {b[TARGET].mean():.1%}")

fig, ax = plt.subplots(figsize=(5, 5))
for name, y, p in [("Hospital A (test)", test_a[TARGET], p_test_a), ("Hospital B", b[TARGET], p_b)]:
    observed, predicted = calibration_curve(y, p, n_bins=6, strategy="quantile")
    ax.plot(predicted, observed, marker="o", label=name)
ax.plot([0, 1], [0, 1], "k--", label="perfectly calibrated")
ax.set_xlabel("predicted risk"); ax.set_ylabel("observed event rate")
ax.legend(); plt.title("Calibration"); plt.show()

**Question:** What does it mean for the physician in hospital B when the system shows “risk 40 %”?

*Your answer:* …

In [ ]:
# Bonus: recalibrate the intercept (logit shift), e.g. with a small sample from hospital B.
# Idea: logit(p_new) = logit(p_old) + delta, where delta is chosen so that the mean matches the rate in B.


## Part 7 – What does a missing value mean? (7 min)

`laktat` (lactate) was not measured for all patients.

**Task 7a:** Compare the share of missing lactate values – separately by outcome – in hospitals A and B.

In [ ]:
# TODO: share of missing lactate values per hospital and outcome
# Hint: df.groupby(TARGET)["laktat"].apply(lambda s: s.isna().mean())


> **Note:** The following model is trained *and* interpreted on the data of hospital B. It serves
> **only to demonstrate the relationship** and is not a valid model evaluation. We are only interested in
> the coefficient of `laktat_fehlt` (lactate missing), not in the predictive performance.

In [ ]:
# A model in hospital B that additionally knows WHETHER lactate was measured
b["laktat_fehlt"] = b["laktat"].isna().astype(int)
model_b = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model_b.fit(b[FEATURES + ["laktat_fehlt"]], b[TARGET])
coef = pd.Series(model_b[-1].coef_[0], index=FEATURES + ["laktat_fehlt"]).sort_values()
coef

**Task 7b:** What sign does the coefficient of `laktat_fehlt` have – and what has the model actually
learned from it? Would this knowledge transfer to hospital A?

*Your answer:* …

## Part 8 – From model to CDSS (5 min)

Fill in the CDSS chain for your system – in keywords:

| Link | Your system |
|---|---|
| **DATA** – which data, how current? | … |
| **TRIGGER** – when does it run (e.g. new vital sign, `patient-view`)? | … |
| **LOGIC** – rule, model or both? Which threshold? | … |
| **OUTPUT** – info, warning, hard stop? To whom? | … |
| **ACTION** – what exactly should the person do? | … |
| **FEEDBACK** – which metrics do you monitor after go-live? | … |

**Final question:** Your notebook is a teaching prototype. What would have to change for it to become a medical
device – and what role does the **intended purpose** play?